In [ ]:
# Placeholder parameters: illustrative values only, not the ones used in live research.
WINDOW = 20  # 個人智慧財產,參數不提供 (placeholder)
Z_ENTRY_ABS = 2.0  # 個人智慧財產,參數不提供 (placeholder)
Z_EXIT_ABS = 0.5  # 個人智慧財產,參數不提供 (placeholder)
LAG_DAYS = 5  # 個人智慧財產,參數不提供 (placeholder)
REGIME_MIN_Z = 1  # 個人智慧財產,參數不提供 (placeholder)
DIVIDEND_YIELD = 0.08  # 個人智慧財產,參數不提供 (placeholder)
RISK_FREE = 0.03
CONVEX_Z = 1.0  # 個人智慧財產,參數不提供 (placeholder)

### X Mount Google Drive

Run the following cell to mount your Google Drive. You will be prompted to authorize Colab to access your Drive files.

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

### X Access a file from Google Drive

Once your Drive is mounted, you can access files using their path, typically starting with `data/`. Replace `'Your_Folder/your_file.csv'` with the actual path to your file in Google Drive.

In [ ]:
import os

folder_path = 'data/bloomberg'

print(f"Listing contents of: {folder_path}")

if os.path.exists(folder_path):
    for root, dirs, files in os.walk(folder_path):
        level = root.replace(folder_path, '').count(os.sep)
        indent = ' ' * 4 * (level)
        print(f'{indent}{os.path.basename(root)}/')
        subindent = ' ' * 4 * (level + 1)
        for f in files:
            print(f'{subindent}{f}')
else:
    print(f"Error: The folder '{folder_path}' does not exist or is not accessible. Please ensure Google Drive is mounted and the path is correct.")

In [ ]:
import pandas as pd
import os

# Define the base folder path
base_folder_path = 'data/bloomberg'
credit_indexes_path = os.path.join(base_folder_path, 'Credit', 'Credit_Indexes')

# Define file paths for HY_Index and IG_Index
hy_index_file = os.path.join(credit_indexes_path, 'HY_Index.xlsx')
ig_index_file = os.path.join(credit_indexes_path, 'IG_Index.xlsx')

# Load the data
try:
    df_hy = pd.read_excel(hy_index_file)
    df_ig = pd.read_excel(ig_index_file)

    # Assuming the date column is named 'Date' in both files and is the first column
    # Adjust column names if they are different in the actual Excel files
    df_hy.columns = ['Date', 'HY_Index']
    df_ig.columns = ['Date', 'IG_Index']

    # Convert 'Date' column to datetime objects
    df_hy['Date'] = pd.to_datetime(df_hy['Date'])
    df_ig['Date'] = pd.to_datetime(df_ig['Date'])

    # Merge the dataframes on the 'Date' column
    df_credit_spread = pd.merge(df_hy, df_ig, on='Date', how='inner')

    # Calculate the credit spread
    df_credit_spread['Credit_Spread'] = df_credit_spread['HY_Index'] - df_credit_spread['IG_Index']

    print("Credit Spread Calculation Complete. First 5 rows:")
    display(df_credit_spread.head())

except FileNotFoundError as e:
    print(f"Error: One of the files was not found. Please check the path. {e}")
except Exception as e:
    print(f"An error occurred during data loading or processing: {e}")

In [ ]:
import pandas as pd

# Load the downloaded CSV file
hy_yield_df = pd.read_csv('/content/BAMLH0A0HYM2.csv')

# Display the first X rows and information about the DataFrame
print("Downloaded HY Yield Data:")
display(hy_yield_df.head())
print("\nDataFrame Info:")
hy_yield_df.info()

In [ ]:
import pandas as pd

# Load the downloaded CSV file for IG Option-Adjusted Spread
ig_spread_df = pd.read_csv('/content/BAMLC0A0CM.csv')

# Display the first X rows and information about the DataFrame
print("Downloaded IG Option-Adjusted Spread Data:")
display(ig_spread_df.head())
print("\nDataFrame Info:")
ig_spread_df.info()

### Process Ares Capital Stock Price History

In [ ]:
import pandas as pd

# Load the Ares Capital Stock Price History CSV file
ares_df = pd.read_csv('/content/Ares Capital Stock Price History.csv')

# Display the first X rows and information about the DataFrame
print("Ares Capital Stock Price History Data:")
display(ares_df.head())
print("\nDataFrame Info:")
ares_df.info()

### Calculate and Plot HY-IG Spread (Yield-based)

In [ ]:
# Ensure 'observation_date' is datetime and rename columns for clarity
hy_yield_df['observation_date'] = pd.to_datetime(hy_yield_df['observation_date'])
hy_yield_df = hy_yield_df.rename(columns={'BAMLH0A0HYM2': 'HY_Effective_Yield'})

ig_spread_df['observation_date'] = pd.to_datetime(ig_spread_df['observation_date'])
ig_spread_df = ig_spread_df.rename(columns={'BAMLC0A0CM': 'IG_OAS'})

# Merge the two dataframes on 'observation_date'
# Use an outer merge to keep all dates, then drop NaNs if a date is missing in either series
merged_spread_df = pd.merge(hy_yield_df, ig_spread_df, on='observation_date', how='outer')
merged_spread_df = merged_spread_df.dropna().sort_values(by='observation_date').reset_index(drop=True)

# Calculate the HY-IG Spread (Effective Yield - OAS)
merged_spread_df['HY_IG_Spread'] = merged_spread_df['HY_Effective_Yield'] - merged_spread_df['IG_OAS']

print("Merged DataFrame with HY-IG Spread:")
display(merged_spread_df.head())
display(merged_spread_df.info())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set the style for the plots
sns.set_style("whitegrid")

plt.figure(figsize=(15, 8))

sns.lineplot(x='observation_date', y='HY_Effective_Yield', data=merged_spread_df, label='HY Effective Yield')
sns.lineplot(x='observation_date', y='IG_OAS', data=merged_spread_df, label='IG Option-Adjusted Spread (OAS)')
sns.lineplot(x='observation_date', y='HY_IG_Spread', data=merged_spread_df, label='HY-IG Spread (Yield - OAS)', linewidth=2.5)

plt.title('High Yield and Investment Grade Spreads Over Time', fontsize=16)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Spread / Yield (%)', fontsize=12)
plt.legend(title='Series', fontsize=10, title_fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

### Rolling Z-Score Analysis of HY-IG Spread

In [ ]:
# Define the rolling window size (e.g., X periods for a month, adjust as needed)
rolling_window = WINDOW

# Calculate rolling mean and standard deviation for 'HY_IG_Spread'
merged_spread_df['HY_IG_Spread_Rolling_Mean'] = merged_spread_df['HY_IG_Spread'].rolling(window=rolling_window).mean()
merged_spread_df['HY_IG_Spread_Rolling_Std'] = merged_spread_df['HY_IG_Spread'].rolling(window=rolling_window).std()

# Calculate the rolling Z-score
merged_spread_df['HY_IG_Spread_Z_Score'] = (merged_spread_df['HY_IG_Spread'] - merged_spread_df['HY_IG_Spread_Rolling_Mean']) / merged_spread_df['HY_IG_Spread_Rolling_Std']

print(f"DataFrame with Rolling Z-Score for a {rolling_window}-period window:")
display(merged_spread_df.tail()) # Display the tail as rolling calculations will have NaNs at the beginning
display(merged_spread_df.info())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

plt.figure(figsize=(15, 8))

sns.lineplot(x='observation_date', y='HY_IG_Spread_Z_Score', data=merged_spread_df)

plt.title(f'Rolling Z-Score of HY-IG Spread (Window = {rolling_window} periods)', fontsize=16)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Z-Score', fontsize=12)
plt.axhline(0, color='gray', linestyle='--', linewidth=1) # Add a horizontal line at Z=X
plt.axhline(2, color='red', linestyle=':', linewidth=1)  # Example: Z = +X (potentially overstretched)
plt.axhline(-2, color='green', linestyle=':', linewidth=1) # Example: Z = -X (potentially undervalued)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

### Align Ares Capital Stock Price with HY-IG Spread and SOFR Convexity

In [ ]:
import pandas as pd

# X. Prepare Convexity for merge
convex_to_merge = curve_df[['Date', 'Convexity']].copy()
convex_to_merge['observation_date'] = pd.to_datetime(convex_to_merge['Date'])

# X. Safely handle ares_df date column
if 'Date' in ares_df.columns:
    ares_df['observation_date'] = pd.to_datetime(ares_df['Date'])
else:
    ares_df['observation_date'] = pd.to_datetime(ares_df['observation_date'])

# X. Merge Ares Capital data with the credit spread data
combined_df = pd.merge(ares_df[['observation_date', 'Price']],
                       merged_spread_df[['observation_date', 'HY_IG_Spread', 'HY_IG_Spread_Z_Score']],
                       on='observation_date', how='inner')

# X. Merge in the Convexity data
combined_df = pd.merge(combined_df, convex_to_merge[['observation_date', 'Convexity']],
                       on='observation_date', how='left')

# Sort and clean
combined_df = combined_df.sort_values(by='observation_date').reset_index(drop=True)

print("Aligned DataFrame with Price, Spread, and Convexity:")
display(combined_df.head())
display(combined_df.info())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

fig, axes = plt.subplots(2, 1, figsize=(15, 12), sharex=True) # Two subplots, sharing the x-axis

# Plot Ares Capital Price
sns.lineplot(x='observation_date', y='Price', data=combined_df, ax=axes[0], color='blue')
axes[0].set_title('Ares Capital Stock Price Over Time', fontsize=16)
axes[0].set_ylabel('Price', fontsize=12)
axes[0].grid(True, linestyle='--', alpha=0.6)

# Plot HY-IG Spread
sns.lineplot(x='observation_date', y='HY_IG_Spread', data=combined_df, ax=axes[1], color='red')
axes[1].set_title('HY-IG Spread Over Time', fontsize=16)
axes[1].set_xlabel('Date', fontsize=12)
axes[1].set_ylabel('Spread (%)', fontsize=12)
axes[1].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

### Plot Aligned Data

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

fig, ax1 = plt.subplots(figsize=(15, 8))

# Plot Price on primary axis
color = 'tab:blue'
ax1.set_xlabel('Date', fontsize=12)
ax1.set_ylabel('Ares Capital Price', color=color, fontsize=12)
sns.lineplot(x='observation_date', y='Price', data=combined_df, ax=ax1, color=color, label='Ares Capital Price')
ax1.tick_params(axis='y', labelcolor=color)

# Plot Z-Score on secondary axis
ax2 = ax1.twinx()
color = 'tab:red'
ax2.set_ylabel('HY-IG Spread Z-Score', color=color, fontsize=12)
sns.lineplot(x='observation_date', y='HY_IG_Spread_Z_Score', data=combined_df, ax=ax2, color=color, label='HY-IG Spread Z-Score')
ax2.tick_params(axis='y', labelcolor=color)

# Reference lines for Z-score
ax2.axhline(0, color='gray', linestyle='--', linewidth=1)
ax2.axhline(2, color='orange', linestyle=':', linewidth=1)
ax2.axhline(-2, color='green', linestyle=':', linewidth=1)

plt.title('Ares Capital Price vs. HY-IG Spread Rolling Z-Score (Aligned Dates)', fontsize=16)
fig.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

fig, ax1 = plt.subplots(figsize=(15, 8))

# Plot Price on primary axis
color = 'tab:blue'
ax1.set_xlabel('Date', fontsize=12)
ax1.set_ylabel('Ares Capital Price ($)', color=color, fontsize=12)
sns.lineplot(x='observation_date', y='Price', data=combined_df, ax=ax1, color=color, label='Ares Capital Price')
ax1.tick_params(axis='y', labelcolor=color)

# Plot Real Spread on secondary axis
ax2 = ax1.twinx()
color = 'tab:purple'
ax2.set_ylabel('HY-IG Spread (%)', color=color, fontsize=12)
sns.lineplot(x='observation_date', y='HY_IG_Spread', data=combined_df, ax=ax2, color='tab:red', label='HY-IG Spread (%)')
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Ares Capital Price vs. Real HY-IG Spread (Aligned Dates)', fontsize=16)
fig.tight_layout()
plt.show()

### X Correlation Analysis: Ares Price vs. HY-IG Spread

In this section, we calculate the statistical correlation to determine the strength and direction of the relationship between the stock price and the credit spread.

In [ ]:
# Calculate the overall Pearson correlation coefficient
overall_corr = combined_df['Price'].corr(combined_df['HY_IG_Spread'])

# Calculate a rolling correlation (e.g., X-day window)
rolling_corr = combined_df['Price'].rolling(window=60).corr(combined_df['HY_IG_Spread'])

print(f"Overall Pearson Correlation: {overall_corr:.4f}")

# Prepare data for multiple correlation analysis
correlation_df = pd.merge(ares_df[['observation_date', 'Price']],
                          merged_spread_df[['observation_date', 'HY_Effective_Yield', 'IG_OAS', 'HY_IG_Spread']],
                          on='observation_date', how='inner')

# Calculate rolling correlations (X-day window)
window = WINDOW
rolling_corr_hy = correlation_df['Price'].rolling(window=window).corr(correlation_df['HY_Effective_Yield'])
rolling_corr_ig = correlation_df['Price'].rolling(window=window).corr(correlation_df['IG_OAS'])
rolling_corr_spread = correlation_df['Price'].rolling(window=window).corr(correlation_df['HY_IG_Spread'])

# Visualize the three rolling correlations in separate subplots
fig, axes = plt.subplots(3, 1, figsize=(15, 15), sharex=True)

# Plot X: Price vs. HY Effective Yield
axes[0].plot(correlation_df['observation_date'], rolling_corr_hy, color='blue', label='Price vs. HY Yield')
axes[0].set_title(f'{window}-Day Rolling Correlation: Ares Price vs. HY Effective Yield')

# Plot X: Price vs. IG OAS
axes[1].plot(correlation_df['observation_date'], rolling_corr_ig, color='green', label='Price vs. IG OAS')
axes[1].set_title(f'{window}-Day Rolling Correlation: Ares Price vs. IG OAS')

# Plot X: Price vs. HY-IG Spread
axes[2].plot(correlation_df['observation_date'], rolling_corr_spread, color='red', label='Price vs. HY-IG Spread')
axes[2].set_title(f'{window}-Day Rolling Correlation: Ares Price vs. HY-IG Spread')

# Formatting all subplots
for ax in axes:
    ax.axhline(0, color='black', linestyle='--', alpha=0.5)
    ax.set_ylim(-1, 1)
    ax.set_ylabel('Correlation')
    ax.grid(True, linestyle=':', alpha=0.6)
    ax.legend(loc='upper left')

plt.xlabel('Date')
plt.tight_layout()
plt.show()

### X Process SOFR Spot and Futures Data

We will load the SOFR spot rate and the hourly futures data from the Google Drive directory structure identified earlier.

In [ ]:
import pandas as pd
import os

# Define paths for SOFR data
sofr_spot_path = 'data/SOFR_rate.xlsx'
sofr_futures_dir = 'data/'

# Load SOFR Spot Rate
try:
    df_sofr_spot = pd.read_excel(sofr_spot_path)
    print("SOFR Spot Rate Data Loaded. First 5 rows:")
    display(df_sofr_spot.head())

    # Load the front-month SOFR Future (SFRX)
    sfr1_path = os.path.join(sofr_futures_dir, 'SFR1_hourly.xlsx')
    df_sfr1 = pd.read_excel(sfr1_path)
    print("\nSOFR Future (SFR1) Hourly Data Loaded. First 5 rows:")
    display(df_sfr1.head())

except Exception as e:
    print(f"An error occurred while loading SOFR data: {e}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import os

# X. Plot Historical SOFR Spot Data
plt.figure(figsize=(15, 6))
sns.lineplot(x='Date', y='Last Price', data=df_sofr_spot.dropna())
plt.title('Historical SOFR Spot Rate (%)', fontsize=16)
plt.ylabel('Rate (%)')
plt.xlabel('Date')
plt.grid(True, alpha=0.3)
plt.show()

# X. Construct and Plot Near-Term SOFR Futures Term Structure
futures_list = ['SFR1', 'SFR2', 'SFR3', 'SFR4', 'SFR5', 'SFR6']
implied_rates = []

for f in futures_list:
    file_path = os.path.join(sofr_futures_dir, f'{f}_hourly.xlsx')
    temp_df = pd.read_excel(file_path)
    # Get the latest price (assuming first row is latest based on previous head() output)
    latest_price = temp_df['Last Price'].iloc[0]
    # Convert Price to Implied Rate
    implied_rate = 100 - latest_price
    implied_rates.append(implied_rate)

plt.figure(figsize=(10, 6))
plt.plot(futures_list, implied_rates, marker='o', linestyle='-', color='red', linewidth=2)
plt.title('SOFR Futures Term Structure (Implied Rates)', fontsize=16)
plt.ylabel('Implied Rate (%)')
plt.xlabel('Contract Month')
plt.grid(True, linestyle='--', alpha=0.7)
for i, rate in enumerate(implied_rates):
    plt.annotate(f'{rate:.3f}%', (futures_list[i], implied_rates[i]), textcoords="offset points", xytext=(0,10), ha='center')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import os

futures_list = ['SFR1', 'SFR2', 'SFR3', 'SFR4', 'SFR5', 'SFR6']
fig, axes = plt.subplots(3, 2, figsize=(18, 15), sharex=True)
axes = axes.flatten()

for i, f in enumerate(futures_list):
    file_path = os.path.join(sofr_futures_dir, f'{f}_hourly.xlsx')
    df_future = pd.read_excel(file_path)
    df_future['Date'] = pd.to_datetime(df_future['Date'])

    # Plotting
    axes[i].plot(df_future['Date'], df_future['Last Price'], label=f, color='teal')
    axes[i].set_title(f'SOFR Future: {f} Price History', fontsize=14)
    axes[i].set_ylabel('Price')
    axes[i].grid(True, alpha=0.3)
    axes[i].legend()

plt.suptitle('SOFR Futures Price Comparison (SFR1 - SFR6)', fontsize=20)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt

# X. Load and prepare all X futures datasets
futures_list = ['SFR1', 'SFR2', 'SFR3', 'SFR4', 'SFR5', 'SFR6']
all_futures_data = []

for f in futures_list:
    file_path = os.path.join(sofr_futures_dir, f'{f}_hourly.xlsx')
    df = pd.read_excel(file_path)
    df['Date'] = pd.to_datetime(df['Date']).dt.date # Use date only for alignment
    df = df.groupby('Date')['Last Price'].mean().reset_index() # Daily average
    df[f'{f}_Yield'] = 100 - df['Last Price']
    all_futures_data.append(df[['Date', f'{f}_Yield']])

# X. Merge into a single curve dataframe
curve_df = all_futures_data[0]
for i in range(1, len(all_futures_data)):
    curve_df = pd.merge(curve_df, all_futures_data[i], on='Date', how='inner')

# X. Calculate Convexity
# We define a simple convexity proxy as the average of (SFR_i-X + SFR_i+X - X*SFR_i)
# This measures the 'bend' in the curve across the X months.
curve_df['Convexity'] = (
    (curve_df['SFR1_Yield'] + curve_df['SFR3_Yield'] - 2 * curve_df['SFR2_Yield']) +
    (curve_df['SFR2_Yield'] + curve_df['SFR4_Yield'] - 2 * curve_df['SFR3_Yield']) +
    (curve_df['SFR3_Yield'] + curve_df['SFR5_Yield'] - 2 * curve_df['SFR4_Yield']) +
    (curve_df['SFR4_Yield'] + curve_df['SFR6_Yield'] - 2 * curve_df['SFR5_Yield'])
) / 4

# X. Plot Daily Convexity
plt.figure(figsize=(15, 6))
plt.plot(curve_df['Date'], curve_df['Convexity'], color='darkviolet', linewidth=1.5)
plt.axhline(0, color='black', linestyle='--', alpha=0.5)
plt.title('Daily Yield Curve Convexity (SOFR Futures SFR1-SFR6)', fontsize=16)
plt.ylabel('Convexity Proxy')
plt.xlabel('Date')
plt.grid(True, alpha=0.3)
plt.show()

display(curve_df[['Date', 'Convexity']].tail())

In [ ]:
# Calculate rolling Z-score for SOFR Convexity
rolling_window = WINDOW

curve_df['Convexity_Rolling_Mean'] = curve_df['Convexity'].rolling(window=rolling_window).mean()
curve_df['Convexity_Rolling_Std'] = curve_df['Convexity'].rolling(window=rolling_window).std()
curve_df['Convexity_Z_Score'] = (curve_df['Convexity'] - curve_df['Convexity_Rolling_Mean']) / curve_df['Convexity_Rolling_Std']

# Visualize the Convexity Z-score
plt.figure(figsize=(15, 6))
plt.plot(curve_df['Date'], curve_df['Convexity_Z_Score'], color='magenta', linewidth=1.5)
plt.axhline(0, color='black', linestyle='--', alpha=0.5)
plt.axhline(2, color='red', linestyle=':', alpha=0.8, label='Upper Bound (+2Z)')
plt.axhline(-2, color='blue', linestyle=':', alpha=0.8, label='Lower Bound (-2Z)')
plt.title(f'Rolling Z-Score of SOFR Convexity ({rolling_window}-Day Window)', fontsize=16)
plt.ylabel('Z-Score')
plt.xlabel('Date')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

display(curve_df[['Date', 'Convexity', 'Convexity_Z_Score']].tail())

In [ ]:
# Snapshot of SOFR Futures Data and Derived Metrics
print("SOFR Futures Curve Data Snapshot:")
display(curve_df[['Date', 'SFR1_Yield', 'SFR2_Yield', 'SFR3_Yield', 'SFR4_Yield', 'SFR5_Yield', 'SFR6_Yield', 'Convexity', 'Convexity_Z_Score']].tail(10))

# Quick summary of the metrics
print("\nSummary Statistics for Convexity Metrics:")
display(curve_df[['Convexity', 'Convexity_Z_Score']].describe())

In [ ]:
import matplotlib.pyplot as plt

# X. Prepare Convexity data for alignment
# curve_df['Date'] is already an object (from .dt.date)
convex_df = curve_df[['Date', 'Convexity']].copy()
convex_df['observation_date'] = pd.to_datetime(convex_df['Date'])

# X. Merge with combined_df (which contains Ares Price)
ares_convex_df = pd.merge(combined_df[['observation_date', 'Price']], convex_df[['observation_date', 'Convexity']], on='observation_date', how='inner')

# X. Plot
fig, ax1 = plt.subplots(figsize=(15, 8))

# Primary Axis: Ares Price
color_price = '#004488' # Darker blue for contrast
ax1.set_xlabel('Date')
ax1.set_ylabel('Ares Capital Price ($)', color=color_price, fontsize=12)
ln1 = ax1.plot(ares_convex_df['observation_date'], ares_convex_df['Price'], color=color_price, label='Ares Price', linewidth=2)
ax1.tick_params(axis='y', labelcolor=color_price)

# Secondary Axis: Convexity
ax2 = ax1.twinx()
color_conv = '#AA3377' # High-contrast magenta/violet
ax2.set_ylabel('Yield Curve Convexity', color=color_conv, fontsize=12)
ln2 = ax2.plot(ares_convex_df['observation_date'], ares_convex_df['Convexity'], color=color_conv, label='Yield Curve Convexity', alpha=0.8, linewidth=1.5)
ax2.axhline(0, color='black', linestyle='--', alpha=0.5)
ax2.tick_params(axis='y', labelcolor=color_conv)

# Combined Legend
lns = ln1 + ln2
labs = [l.get_label() for l in lns]
ax1.legend(lns, labs, loc='upper left', frameon=True, shadow=True)

plt.title('Ares Capital Price vs. Yield Curve Convexity', fontsize=16, fontweight='bold')
ax1.grid(True, linestyle=':', alpha=0.6)
fig.tight_layout()
plt.show()

# Correlation
conv_corr = ares_convex_df['Price'].corr(ares_convex_df['Convexity'])
print(f"Correlation between Ares Price and Yield Curve Convexity: {conv_corr:.4f}")

In [ ]:

# X. Prepare SOFR data for alignment
df_sofr_plot = df_sofr_spot.dropna().copy()
df_sofr_plot['Date'] = pd.to_datetime(df_sofr_plot['Date'])
df_sofr_plot = df_sofr_plot.rename(columns={'Date': 'observation_date', 'Last Price': 'SOFR_Spot'})

# X. Merge with existing combined_df (which already has Ares Price and Credit Spreads)
# Note: combined_df observation_date is already datetime
sofr_ares_df = pd.merge(combined_df, df_sofr_plot, on='observation_date', how='inner')

# X. Plot Ares Price vs SOFR Spot
fig, ax1 = plt.subplots(figsize=(15, 8))

# Primary Axis: Ares Capital Price
color_price = 'tab:blue'
ax1.set_xlabel('Date')
ax1.set_ylabel('Ares Capital Price ($)', color=color_price, fontsize=12)
ax1.plot(sofr_ares_df['observation_date'], sofr_ares_df['Price'], color=color_price, label='Ares Price')
ax1.tick_params(axis='y', labelcolor=color_price)

# Secondary Axis: SOFR Spot Rate
ax2 = ax1.twinx()
color_sofr = 'tab:orange'
ax2.set_ylabel('SOFR Spot Rate (%)', color=color_sofr, fontsize=12)
ax2.plot(sofr_ares_df['observation_date'], sofr_ares_df['SOFR_Spot'], color=color_sofr, label='SOFR Spot', linestyle='--')
ax2.tick_params(axis='y', labelcolor=color_sofr)

plt.title('Ares Capital Price vs. SOFR Spot Rate', fontsize=16)
ax1.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

# Calculate Correlation
sofr_corr = sofr_ares_df['Price'].corr(sofr_ares_df['SOFR_Spot'])
print(f"Correlation between Ares Price and SOFR Spot: {sofr_corr:.4f}")


### X Backtesting Strategy: HY-IG Spread Z-Score Signals

We will define a simple signal-based strategy:
- **Enter Long:** Z-Score < -X
- **Exit/Neutral:** Z-Score > X

*Note: This is a simplified illustrative backtest and does not account for transaction costs or slippage.*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- PARAMETERS ---
Z_ENTRY = Z_ENTRY_ABS
Z_EXIT = Z_EXIT_ABS
LAG = LAG_DAYS
REGIME_THRESHOLD = REGIME_MIN_Z
ANNUAL_DIVIDEND = DIVIDEND_YIELD
DAILY_DIVIDEND = ANNUAL_DIVIDEND / 252

# --- PREP DATA ---
backtest_df = combined_df.dropna(subset=['HY_IG_Spread_Z_Score']).copy()

# Lagged Z-score
backtest_df['Z_lag'] = backtest_df['HY_IG_Spread_Z_Score'].shift(LAG)

# --- SIGNAL GENERATION (LONG ONLY) ---
backtest_df['Signal'] = 0

# LONG: spreads tight (Z low)
# Note: In our current Z-score, a negative Z means the spread is tighter than rolling average
long_cond = (backtest_df['Z_lag'] < -Z_ENTRY) & (abs(backtest_df['Z_lag']) > REGIME_THRESHOLD)

# Set signal to X for Long, X for Cash
backtest_df.loc[long_cond, 'Signal'] = 1

# --- POSITION MANAGEMENT ---
# Forward fill the long signal
backtest_df['Position'] = backtest_df['Signal'].replace(0, np.nan).ffill()

# Exit condition: Z crosses zero or specified exit threshold
exit_cond = (backtest_df['HY_IG_Spread_Z_Score'] > -Z_EXIT)
backtest_df.loc[exit_cond, 'Position'] = 0
backtest_df['Position'] = backtest_df['Position'].fillna(0)

# --- RETURNS CALCULATION ---
backtest_df['Market_Price_Return'] = backtest_df['Price'].pct_change()
backtest_df['Market_Total_Return'] = backtest_df['Market_Price_Return'] + DAILY_DIVIDEND

# Strategy Return = Position * Total Market Return (Includes price move and dividend while held)
backtest_df['Strategy_Returns'] = backtest_df['Position'].shift(1) * backtest_df['Market_Total_Return']

# --- CUMULATIVE ---
backtest_df['Cumulative_Market'] = (1 + backtest_df['Market_Total_Return'].fillna(0)).cumprod()
backtest_df['Cumulative_Strategy'] = (1 + backtest_df['Strategy_Returns'].fillna(0)).cumprod()

# --- PLOT ---
plt.figure(figsize=(15, 8))
plt.plot(backtest_df['observation_date'], backtest_df['Cumulative_Market'], label='ARCC Buy & Hold (Total Return)', alpha=0.5, linestyle='--')
plt.plot(backtest_df['observation_date'], backtest_df['Cumulative_Strategy'], label='Long-Only Strategy Return', linewidth=2, color='teal')
plt.title('Backtest: Long-Only Credit Spread Strategy vs. ARCC Total Return', fontsize=16)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# --- PERFORMANCE ---
total_return = (backtest_df['Cumulative_Strategy'].iloc[-1] - 1) * 100
bnh_return = (backtest_df['Cumulative_Market'].iloc[-1] - 1) * 100
print(f"Total Long-Only Strategy Return: {total_return:.2f}%")
print(f"Buy & Hold Total Return: {bnh_return:.2f}%")

In [ ]:
# Identify entries specifically for the combined plot
backtest_convex_df = pd.merge(backtest_df, curve_df[['Date', 'Convexity_Z_Score']], left_on='observation_date', right_on=pd.to_datetime(curve_df['Date']), how='left')

backtest_convex_df['Entry'] = backtest_convex_df['Position'].diff().fillna(0)
long_entries_convex = backtest_convex_df[backtest_convex_df['Entry'] > 0]
short_entries_convex = backtest_convex_df[backtest_convex_df['Entry'] < 0]

plt.figure(figsize=(15, 8))

# Plot Cumulative Returns
plt.plot(backtest_convex_df['observation_date'], backtest_convex_df['Cumulative_Market'],
         label='Ares Buy & Hold', color='gray', alpha=0.4, linestyle='--')
plt.plot(backtest_convex_df['observation_date'], backtest_convex_df['Cumulative_Strategy'],
         label='Strategy Return', color='green', linewidth=2)

# Add markers for trade timing
plt.scatter(long_entries_convex['observation_date'], long_entries_convex['Cumulative_Strategy'],
            marker='^', color='blue', s=120, label='Long Entry', zorder=5, edgecolors='white')
plt.scatter(short_entries_convex['observation_date'], short_entries_convex['Cumulative_Strategy'],
            marker='v', color='red', s=120, label='Short Entry', zorder=5, edgecolors='white')

# Background Shading for Convexity Z-Scores
pos_z_mask = backtest_convex_df['Convexity_Z_Score'] > CONVEX_Z
neg_z_mask = backtest_convex_df['Convexity_Z_Score'] < -CONVEX_Z

plt.fill_between(backtest_convex_df['observation_date'], 0.5, 2.0,
                 where=pos_z_mask, color='blue', alpha=0.07, label='Positive Z-Score (>0.5)')
plt.fill_between(backtest_convex_df['observation_date'], 0.5, 2.0,
                 where=neg_z_mask, color='red', alpha=0.07, label='Negative Z-Score (<-0.5)')

plt.title('Trade Performance vs. SOFR Convexity Z-Score Regimes', fontsize=16)
plt.ylabel('Cumulative Return')
plt.xlabel('Date')
plt.legend(loc='upper left', ncol=2)
plt.ylim(backtest_convex_df['Cumulative_Market'].min() * 0.9, backtest_convex_df['Cumulative_Strategy'].max() * 1.1)
plt.grid(True, alpha=0.2)
plt.show()

### X Credit Cycle Regime Classification
We define the 'Credit Cycle' by looking at the absolute level of the HY-IG Spread relative to its history and the trend of SOFR.

In [ ]:
import pandas as pd
import numpy as np

# X. Use the merged spread data to find thresholds
spread_data = merged_spread_df['HY_IG_Spread'].dropna()
low_threshold = spread_data.quantile(0.25)
high_threshold = spread_data.quantile(0.75)

# X. Define a function to classify regimes
def classify_regime(row):
    spread = row['HY_IG_Spread']
    if spread <= low_threshold:
        return 'Expansion (Low Risk)'
    elif spread >= high_threshold:
        return 'Distress/Contraction'
    else:
        return 'Mid-Cycle/Transition'

# X. Apply to our backtest dataframe
backtest_df['Cycle_Regime'] = backtest_df.apply(classify_regime, axis=1)

# X. Analyze performance by regime
regime_perf = backtest_df.groupby('Cycle_Regime')['Strategy_Returns'].agg(['mean', 'std', 'count'])
regime_perf['Sharpe_Proxy'] = (regime_perf['mean'] / regime_perf['std']) * np.sqrt(252)

print(f"Spread Thresholds: Low (25th): {low_threshold:.2f}%, High (75th): {high_threshold:.2f}%")
display(regime_perf)

# X. Visualize the current regime over time
plt.figure(figsize=(15, 3))
colors = {'Expansion (Low Risk)': 'green', 'Mid-Cycle/Transition': 'orange', 'Distress/Contraction': 'red'}
for regime in colors:
    mask = backtest_df['Cycle_Regime'] == regime
    plt.scatter(backtest_df.loc[mask, 'observation_date'], [1]*mask.sum(), c=colors[regime], label=regime, marker='|', s=500)

plt.title('Historical Credit Cycle Regimes (Based on HY-IG Spread)')
plt.yticks([])
plt.legend(loc='upper right', bbox_to_anchor=(1.15, 1))
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

fig, ax1 = plt.subplots(figsize=(15, 8))

# X. Plot Cumulative Strategy Returns on primary axis
ax1.set_xlabel('Date', fontsize=12)
ax1.set_ylabel('Cumulative Total Return (Growth of $1)', fontsize=12)

# Plot Strategy
ax1.plot(backtest_df['observation_date'], backtest_df['Cumulative_Strategy'],
         color='green', linewidth=2.5, label='Active Strategy Return')

# Plot Buy & Hold
ax1.plot(backtest_df['observation_date'], backtest_df['Cumulative_Market'],
         color='blue', linewidth=1.5, linestyle='--', label='ARCC Buy & Hold (Total Return)')

ax1.tick_params(axis='y')

# X. Add Background Shading for Credit Cycle Regimes
regime_colors = {'Expansion (Low Risk)': 'green', 'Mid-Cycle/Transition': 'orange', 'Distress/Contraction': 'red'}

# Group consecutive regimes for cleaner shading
backtest_df['regime_change'] = (backtest_df['Cycle_Regime'] != backtest_df['Cycle_Regime'].shift()).cumsum()
regime_groups = backtest_df.groupby(['regime_change', 'Cycle_Regime'])['observation_date'].agg(['min', 'max'])

for (change_idx, regime_name), row in regime_groups.iterrows():
    ax1.axvspan(row['min'], row['max'], color=regime_colors[regime_name], alpha=0.1)

# X. Create Legend
legend_elements = [
    Line2D([0], [0], color='green', lw=2.5, label='Active Strategy PnL'),
    Line2D([0], [0], color='blue', lw=1.5, linestyle='--', label='ARCC Buy & Hold'),
    Line2D([0], [0], color='green', alpha=0.3, marker='s', linestyle='None', label='Expansion (Regime)'),
    Line2D([0], [0], color='orange', alpha=0.3, marker='s', linestyle='None', label='Mid-Cycle (Regime)'),
    Line2D([0], [0], color='red', alpha=0.3, marker='s', linestyle='None', label='Distress (Regime)')
]
ax1.legend(handles=legend_elements, loc='upper left', frameon=True, shadow=True)

plt.title('Strategy Performance vs. Buy & Hold Across Credit Cycle Regimes', fontsize=16, fontweight='bold')
ax1.grid(True, linestyle=':', alpha=0.4)
fig.tight_layout()
plt.show()

### X Long-Only Backtest Implementation
This strategy only takes long positions in ARCC when credit conditions are exceptionally favorable (Tight Spreads) and holds cash otherwise.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# --- CONFIGURABLE PARAMETERS ---
START_DATE = "2023-01-01"  # Change this to your desired start date (YYYY-MM-DD)
Z_ENTRY = Z_ENTRY_ABS
Z_EXIT = Z_EXIT_ABS
LAG = LAG_DAYS
ANNUAL_DIVIDEND = DIVIDEND_YIELD
DAILY_DIVIDEND = ANNUAL_DIVIDEND / 252
ANNUAL_RFR = RISK_FREE
DAILY_RFR = ANNUAL_RFR / 252

# --- ALIGN DATA ---
# Using existing dataframes from memory
convex_to_merge = curve_df[['Date', 'Convexity']].copy()
convex_to_merge['observation_date'] = pd.to_datetime(convex_to_merge['Date'])

# Handle Ares data date format
ares_temp = ares_df.copy()
if 'Date' in ares_temp.columns:
    ares_temp['observation_date'] = pd.to_datetime(ares_temp['Date'])
else:
    ares_temp['observation_date'] = pd.to_datetime(ares_temp['observation_date'])

# Create combined dataset for backtesting
combined_bt = pd.merge(ares_temp[['observation_date', 'Price']],
                       merged_spread_df[['observation_date', 'HY_IG_Spread', 'HY_IG_Spread_Z_Score']],
                       on='observation_date', how='inner')
combined_bt = pd.merge(combined_bt, convex_to_merge[['observation_date', 'Convexity']],
                       on='observation_date', how='left')
combined_bt = combined_bt.sort_values(by='observation_date').reset_index(drop=True)

# --- PREPARE BACKTEST DATA ---
lo_df = combined_bt.dropna(subset=['HY_IG_Spread_Z_Score']).copy()
lo_df['observation_date'] = pd.to_datetime(lo_df['observation_date'])

# Apply custom start date filter
lo_df = lo_df[lo_df['observation_date'] >= START_DATE].reset_index(drop=True)

# Lagged Z-score for signal execution
lo_df['Z_lag'] = lo_df['HY_IG_Spread_Z_Score'].shift(LAG)

# --- SIGNAL & POSITION ---
lo_df['Signal'] = 0
lo_df.loc[lo_df['Z_lag'] < Z_ENTRY, 'Signal'] = 1
lo_df['Position'] = lo_df['Signal'].replace(0, np.nan).ffill()
lo_df.loc[lo_df['HY_IG_Spread_Z_Score'] > Z_EXIT, 'Position'] = 0
lo_df['Position'] = lo_df['Position'].fillna(0)

# --- RETURNS ---
lo_df['Market_Total_Return'] = lo_df['Price'].pct_change() + DAILY_DIVIDEND
# Earn ARCC total return when long, else earn Risk Free Rate
lo_df['Strategy_Returns'] = np.where(lo_df['Position'].shift(1) == 1, lo_df['Market_Total_Return'], DAILY_RFR)

lo_df['Cum_Market'] = (1 + lo_df['Market_Total_Return'].fillna(0)).cumprod()
lo_df['Cum_Strategy'] = (1 + lo_df['Strategy_Returns'].fillna(0)).cumprod()

# --- VISUALIZATION ---
plt.figure(figsize=(15, 7))
plt.plot(lo_df['observation_date'], lo_df['Cum_Market'], label='ARCC Buy & Hold (Total Return)', color='gray', alpha=0.6)
plt.plot(lo_df['observation_date'], lo_df['Cum_Strategy'], label=f'Long-Only Strategy (Start: {START_DATE})', color='teal', linewidth=2)
plt.title(f'Long-Only Strategy Performance vs. Buy & Hold (Starting {START_DATE})')
plt.ylabel('Growth of $1')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Final Strategy Return (with RFR): {(lo_df['Cum_Strategy'].iloc[-1]-1)*100:.2f}%")
print(f"Final Buy & Hold Return: {(lo_df['Cum_Market'].iloc[-1]-1)*100:.2f}%")